# To - Do
## Bugs:
- Issues and bugs: Censoring only on Halpha and Hbeta

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from robusta_hmf import Robusta
import time
#%matplotlib widget

In [ ]:
#define constants
c = 299792.458 # km/s
LN10 = np.log(10.)
MAX_IVAR = 2.5e3
MIN_IVAR = 0.1
H_ALPHA, H_BETA = 6564.614, 4862.721 #Reference: from classic.sdss.org
HE_I_6680 = 6680 #Reference: Johanna brain
HE_I_4026 = 4026
HE_I_4471 = 4471
HE_I_4922 = 4922
HE_I_4713 = 4713
HE_II_4686 = 4686 #Reference: Johanna brain
HE_II_4542 = 4542
HE_II_4200 = 4200


#lines
lines_balmer = [H_ALPHA, H_BETA]
lines_HE_I = [HE_I_6680, HE_I_4026, HE_I_4471, HE_I_4922, HE_I_4713]
lines_HE_II = [HE_II_4686, HE_II_4542, HE_II_4200]
all_lines = np.concatenate([lines_balmer, lines_HE_I, lines_HE_II])


#line labes
line_labels_balmer = ["Halpha", "Hbeta"]
line_labels_HE_I = ["HeI_6680", "HeI_4026", "HeI_4471", "HeI_4922", "HeI_4713"]
line_labels_HE_II = ["HeII_4686", "HeII_4542", "HeII_4200"]
all_line_labels = np.concatenate([line_labels_balmer, line_labels_HE_I, line_labels_HE_II])

statistics = ["EW", "shift", "width"]
moment_names = ["EW", "M1", "M2", "M3", "M4"]


H_delta_lnlam_line = 1000 / c 
H_delta_loglam_line = H_delta_lnlam_line / LN10 
HE_delta_lnlam_line = 500 / c
HE_delta_loglam_line = HE_delta_lnlam_line / LN10
# log_H_ALPHA, log_H_BETA, log_HE_I, log_HE_II = np.log10(H_ALPHA), np.log10(H_BETA), np.log10(HE_I), np.log10(HE_II)

#delta loglam line for labeling
delta_balmer = np.zeros(len(lines_balmer)) + (1000 / c) / LN10 
delta_HE_I = np.zeros(len(lines_HE_I)) + (500 / c) / LN10 
delta_HE_II = np.zeros(len(lines_HE_II)) + (500 / c) / LN10 
all_delta_loglams = np.concatenate([delta_balmer, delta_HE_I, delta_HE_II])


#indexing is super important for these three arrays 
print(all_lines.shape, all_line_labels.shape, all_delta_loglams.shape)
print(all_lines, all_line_labels, all_delta_loglams)

H_delta_lnlam_line = 1000 / c 
H_delta_loglam_line = H_delta_lnlam_line / LN10 
HE_delta_lnlam_line = 500 / c
HE_delta_loglam_line = HE_delta_lnlam_line / LN10
log_H_ALPHA, log_H_BETA = np.log10(H_ALPHA), np.log10(H_BETA)


half_wids = [H_delta_loglam_line, H_delta_loglam_line, HE_delta_loglam_line, HE_delta_loglam_line] 

In [ ]:
#load stellar parameters
DATESTR = "2026-07-20"

stars = pd.read_parquet(f"parent_stars10k_{DATESTR}.parquet")
print("num of stars in parquet:", stars.shape[0])

spectra_temp = pd.read_parquet(f"spectra_{DATESTR}_table.parquet")
print("num of spectra in parquet:", stars.shape[0])

print(stars.shape)

In [ ]:
print("spectra before:", spectra_temp.shape)
spectra = spectra_temp.join(stars[["GAIA_ID", "Teff_fit", "NANA_HASH"]].set_index("GAIA_ID"), on = "GAIA_ID", rsuffix = "_stars", how = "left", validate = "m:1")
print("spectra after:", spectra.shape)

In [ ]:
print(spectra.columns.to_list())

In [ ]:
nana = "nana_"
all_lines = np.array(all_lines)
all_lines = all_line_labels
for l, line in enumerate(all_lines):
    line
    col_name = nana + line
    print(col_name)
    for s, stat in enumerate(moment_names):
        ##add column here
            # print(col_name + "_" + stat)
            # print(col_name + "_" + stat + "_err")
            # print(col_name + "_" + stat + "_resid")
            # print(col_name + "_" + stat + "_resid_err")
            spectra[col_name + "_" + stat] = 0.
            spectra[col_name + "_" + stat + "_err"] = np.inf
            spectra[col_name + "_" + stat + "_resid"] = 0.
            spectra[col_name + "_" + stat + "_resid_err"] = 0.

print(spectra.columns.to_list())

In [ ]:
# sanity check contents of file
print(spectra[["GAIA_ID", "NANA_HASH", "Teff_fit", "EWobs_Halpha", "rv_hydrogen_all_mean", "SPEC_FILE"]])

In [ ]:
#read in the data and deal with radial velocities

#take the ALL spectra (like 35,000)
with open(f'spectra_{DATESTR}_data.pkl','rb') as f:
    pkl_data = pickle.load(f)
print(len(pkl_data))

In [ ]:
#pkl of ALL spectra (Teff > 10000)
# BUG: This code is very brittle
fluxes, loglam, ivars, continuua, _, _, _, _, _ = pkl_data
print(fluxes.shape, loglam.shape, ivars.shape, continuua.shape)

In [ ]:
# hard stop if we SUCK
assert len(fluxes) == len(spectra)

In [ ]:
# shift to rest frame.
# Note: Only does integer-pixel shifts
# Note: Data have been slightly corrupted; delta_log_lam is not consistent across the wavelength grid.

def get_delta_log_lam(loglambdas):
    return np.median(loglambdas[1:] - loglambdas[:-1])
    
def pixel_shift(flxs, ivrs, contins, rvs, log_lambda):

    delta_log_lambda_pixel = get_delta_log_lam(log_lambda)

    # Catch bad rv values, goddammit
    bad = np.logical_not(np.isfinite(rvs))
    rad_vels = rvs.copy()
    rad_vels[bad] = 0.

    delta_log_lambdas = (rad_vels / c) / LN10
    delta_pixels = np.round(delta_log_lambdas/delta_log_lambda_pixel).astype(int)

    rest_flxs = np.zeros_like(flxs) + 1.
    rest_ivrs = np.zeros_like(ivrs)
    rest_contins = np.zeros_like(contins) + np.nan

    for i, dp in enumerate(delta_pixels):
        if dp < 0:
            rest_flxs[i, -dp:] = flxs[i, :dp]
            rest_ivrs[i, -dp:] = ivrs[i, :dp]
            rest_contins[i, -dp:] = contins[i, :dp]
            
        elif dp > 0:
            rest_flxs[i, :-dp] = flxs[i, dp:]
            rest_ivrs[i, :-dp] = ivrs[i, dp:]
            rest_contins[i, :-dp] = contins[i, dp:]
            
        else:
            rest_flxs[i, :] = flxs[i, :]
            rest_ivrs[i, :] = ivrs[i, :]
            rest_contins[i, :] = contins[i, :]

    # dammit
    rest_ivrs[bad, :] = 0.
    print("zeroing out ivars on", np.sum(bad), "stars with bad rvs")
    return rest_flxs, rest_ivrs, rest_contins

In [ ]:
# shift everything to the rest frame
rest_fluxes, rest_ivars, rest_continuua = pixel_shift(fluxes, ivars, continuua, spectra["rv_hydrogen_all_mean"].to_numpy(), loglam)
print(rest_fluxes.shape)

In [ ]:
def fix_nans_and_infinites(rest_flxs, rest_ivrs):

    #fix nans and infinities
    bad = np.logical_not(np.isfinite(rest_flxs))
    rest_flxs[bad] = 1
    rest_ivrs[bad] = 0
    
    bad = np.logical_not(np.isfinite(rest_ivrs))
    rest_flxs[bad] = 1
    rest_ivrs[bad] = 0
    
    bad = np.logical_or((rest_flxs > 2.0), (rest_flxs < 0))
    rest_flxs[bad] = 1
    rest_ivrs[bad] = 0
    
    bad = rest_ivrs > MAX_IVAR
    # rest_fluxes[bad] = 1
    # rest_ivars[bad] = 0
    rest_ivrs[bad] = MAX_IVAR
    
    bad = rest_ivrs < MIN_IVAR
    rest_flxs[bad] = 1.0
    rest_ivrs[bad] = MIN_IVAR

    return rest_flxs, rest_ivrs

In [ ]:
data, weights = fix_nans_and_infinites(rest_fluxes, rest_ivars)
print(data.shape, weights.shape, rest_continuua.shape)

In [ ]:
# make a sanity plot
tempposts = np.exp(np.linspace(np.log(np.min(spectra["Teff_fit"].to_numpy())), np.log(np.max(spectra["Teff_fit"].to_numpy())), 16))
f = plt.figure(figsize=(12, 12))
for t, T in enumerate(tempposts):
    # find closest high SNR spectrum
    highsnr = np.arange(len(spectra))[np.median(rest_ivars[:, :2000], axis=1) > 500.0]
    ii = np.argmin(np.abs(spectra["Teff_fit"].to_numpy()[highsnr] - T))
    ii = highsnr[ii]
    Teff = spectra["Teff_fit"].iloc[ii]
    print(t, T, ii, Teff)
    # plot it
    plt.step(10. ** loglam, rest_fluxes[ii] + t, c="k", where="mid")
    gid = spectra["GAIA_ID"].iloc[ii]
    label = f"{gid}; Teff = {Teff:5.0f}"
    plt.text(4000., 1.1 + t, label)
plt.ylim(0., len(tempposts) + 1.)
plt.xlim(4000, 5000)
#plt.xlim(H_ALPHA - 200, H_ALPHA + 200)

lines_balmer = [H_ALPHA, H_BETA]
lines_HE_I = [HE_I_6680, HE_I_4026, HE_I_4471, HE_I_4922, HE_I_4713]
lines_HE_II = [HE_II_4686, HE_II_4542, HE_II_4200]

for l, line in enumerate(lines_balmer):
    if l == 0:
        plt.axvline(line, label = "Balmer lines", lw=1, alpha=0.5, zorder=-10, color = "green")
    else:
        plt.axvline(line, lw=1, alpha=0.5, zorder=-10, color = "green")
        
for l, line in enumerate(lines_HE_I):
    if l == 0: 
        plt.axvline(line, label = "HeI lines", lw=1, alpha=0.5, zorder=-10, color = "orange")
    else:
        plt.axvline(line, lw=1, alpha=0.5, zorder=-10, color = "orange")
        
for l, line in enumerate(lines_HE_II):
    if l == 0: 
        plt.axvline(line, label = "HeII lines", lw=1, alpha=0.3, zorder=-10, color = "red")
    
    plt.axvline(line, lw=1, alpha=0.3, zorder=-10, color = "red")

plt.legend()    
plt.title("selected high SNR example stars, ordered by Teff")

In [ ]:
#baddness of fit of fit cut < 2.0
#NANA BOF LIMIT
#NANA EW LIMIT 1.0

In [ ]:
temps = [10000, 15000, 23000, 25000]
temp_labels = ["10K", "15K", "23K", "25K"]

def make_temp_binning(temp_ranges, labels, rest_flxs, rest_ivrs, rest_contins, spectra_df):

    #cut on Johanna's and Nanas' ews
    ews_j = spectra_df['EWobs_Halpha'].to_numpy()
    ews_n = spectra_df['nana_Halpha_EW'].to_numpy()
    good_ew_mask = (ews_j > 0) & (ews_n < 1)
    print(f"Johanna and Nana EW: Keeping {good_ew_mask.sum()} / {len(good_ew_mask)} spectra (removed {(~good_ew_mask).sum()})")
    rest_flxs, rest_ivrs, rest_contins, nana_hash, spectra_temps = rest_flxs[good_ew_mask], rest_ivrs[good_ew_mask], rest_contins[good_ew_mask], spectra_df["NANA_HASH"].to_numpy()[good_ew_mask], spectra_df["Teff_fit"].to_numpy()[good_ew_mask]



    ret_vals = {}
    
    for label, temp in zip(labels, temp_ranges):
        good_temp_mask = temp <= spectra_temps

        
        ret_vals[label] = {
            "flux": rest_flxs[good_temp_mask],
            "ivar": rest_ivrs[good_temp_mask],
            "continuua": rest_contins[good_temp_mask],
            "NANA_HASH": nana_hash[good_temp_mask],
            "count": int(good_temp_mask.sum()),
        }
        
        print(f"Number of spectra with temp >= {temp} is {ret_vals[label]['count']}")
    return ret_vals

In [ ]:
temp_binning_data = make_temp_binning(temps, temp_labels, data, weights, rest_continuua, spectra)

rest_fluxes10K = temp_binning_data["10K"]["flux"]
rest_ivars10K = temp_binning_data["10K"]["ivar"]
nana_hash10K = temp_binning_data["10K"]["NANA_HASH"]

rest_fluxes15K = temp_binning_data["15K"]["flux"]
rest_ivars15K = temp_binning_data["15K"]["ivar"]
nana_hash15K = temp_binning_data["15K"]["NANA_HASH"]


rest_fluxes25K = temp_binning_data["25K"]["flux"]
rest_ivars25K = temp_binning_data["25K"]["ivar"]
nana_hash25K = temp_binning_data["25K"]["NANA_HASH"]

print(rest_fluxes10K.shape, rest_fluxes15K.shape, rest_fluxes25K.shape)
print(nana_hash10K.shape, nana_hash15K.shape, nana_hash25K.shape)

In [ ]:
def cut_on_JohannaEW(rest_flxes, rest_ivrs, rest_contin, spectra_df):
    
    ews = spectra_df['EWobs_Halpha'].to_numpy()
    nana_hash = spectra_df['NANA_HASH'].to_numpy()
    good_ew_mask = ews > 0
    print(f"Johanna EW: Keeping {good_ew_mask.sum()} / {len(good_ew_mask)} spectra (removed {(~good_ew_mask).sum()} with negative EW)")

    return rest_flxes[good_ew_mask], rest_ivrs[good_ew_mask], rest_contin[good_ew_mask], nana_hash[good_ew_mask]
    
def cut_on_NanaEW(rest_flxes, rest_ivrs, rest_contin, spectra_df): 
    ews = spectra_df['nana_Halpha_EW'].to_numpy()
    nana_hash = spectra_df['NANA_HASH'].to_numpy()
    good_ew_mask = ews < 1
    
    print(f"Nana EW: Keeping {good_ew_mask.sum()} / {len(good_ew_mask)} spectra (removed {(~good_ew_mask).sum()} with EW > 1)")
    return rest_flxes[good_ew_mask], rest_ivrs[good_ew_mask], rest_contin[good_ew_mask], nana_hash[good_ew_mask]


In [ ]:
# split data to A and B using nana_hash parity, only train on A
def split_and_train(rest_flxs, rest_ivrs, n_hash, K = 12, scale = 1, nu = 1):
    
    N, M = rest_flxs.shape
    
    Ainx = np.where(n_hash % 2 == 0)[0]
    Binx = np.where(n_hash % 2 == 1)[0]

    print(f"A: {len(Ainx)} spectra, B: {len(Binx)} spectra")

    modA = Robusta(rank=K, robust=True, robust_scale = scale, robust_nu = nu)
    start = time.perf_counter()
    modA.fit(rest_flxs[Ainx], rest_ivrs[Ainx], max_iter=10000)
    end = time.perf_counter()
    print("total time:", (end - start)/60, "minutes")

    modB = Robusta(rank=K, robust=True, robust_scale = scale, robust_nu = nu)
    start = time.perf_counter()
    modB.fit(rest_flxs[Binx], rest_ivrs[Binx], max_iter=10000)
    end = time.perf_counter()
    print("total time:", (end - start)/60, "minutes")


    return modA, modB, Ainx, Binx    

In [ ]:
modelA25K, modelB25K, Aindx25K, Bindx25K = split_and_train(rest_fluxes25K, rest_ivars25K, nana_hash25K)

In [ ]:
def censor_and_synthesize(lines, log_lambdas, rest_flxes, rest_ivrs, Binx, modA, Ainx, modB):

    log_lines = np.log10(lines)
    
    log_H_ALPHA, log_H_BETA = log_lines[0], log_lines[1]
    
    region_alpha = np.abs(log_lambdas - log_H_ALPHA)
    region_beta = np.abs(log_lambdas - log_H_BETA)
    
    censor_mask = np.ones_like(log_lambdas)
    censor_mask[(region_alpha < H_delta_loglam_line) | (region_beta < H_delta_loglam_line) ] = 0 #?
    
    state, _ = modA.infer(rest_flxes[Binx], rest_ivrs[Binx] * censor_mask[None, :])
    synth_B = modA.synthesize(state)

    state, _ = modB.infer(rest_flxes[Ainx], rest_ivrs[Ainx] * censor_mask[None, :])
    synth_A = modB.synthesize(state)

    return synth_A, synth_B

In [ ]:
synth_A25k, synth_B25K = censor_and_synthesize(lines_balmer, loglam, rest_fluxes25K, rest_ivars25K, Bindx25K, modelA25K, Aindx25K, modelB25K)

In [ ]:
modelA15K, modelB15K, Aindx15K, Bindx15K = split_and_train(rest_fluxes15K, rest_ivars15K, nana_hash15K)

In [ ]:
modelA10K, modelB10K, Aindx10K, Bindx10K = split_and_train(rest_fluxes10K, rest_ivars10K, nana_hash10K)

In [ ]:
synth_A15k, synth_B15K = censor_and_synthesize(lines_balmer, loglam, rest_fluxes15K, rest_ivars15K, Bindx15K, modelA15K, Aindx15k, modelB15k)
synth_A10k, synth_B10K = censor_and_synthesize(lines_balmer, loglam, rest_fluxes10K, rest_ivars10K, Bindx10K, modelA10K, Aindx10k, modelB10k)

In [ ]:
def get_moment_vals(log_lambda, rest_flxes, rest_ivars, Binx, synthB):
    
    delta_log_lambda_pixel = get_delta_log_lam(log_lambda)
    N, M = rest_flxes[Binx].shape


    lam = 10 ** log_lambda

    exponents = np.arange(5)
    moments = np.zeros((N, len(lines), len(exponents))) + np.nan
    moment_errs = np.zeros((N, len(lines), len(exponents))) + np.nan

    for j, (line, wid) in enumerate(zip(lines, half_wids)):
        logline = np.log10(line)
        integration_weight = delta_log_lambda_pixel * LN10 * lam * (np.abs(logline - log_lambda) < wid)
        resid = rest_flxes[Binx] - synthB
        diff = lam - line #this is hogg's lam-lam0 he wrote about
    
        #compute vals and corresponding errors
        for k, expo in enumerate(exponents):
            something = integration_weight * diff ** expo
            moments[:, j, k] = np.sum(resid * something[None, :], axis=1)
            moment_errs[:, j, k] = np.sqrt(np.sum(something[None, :] ** 2 / rest_ivars[Binx], axis=1))

    return moments, moment_errs

In [ ]:
moments_10K, moment_errs_10K = get_moment_vals(loglam, rest_fluxes10K, rest_ivars10K, Bindx10K, synth_B10K)
moments_15K, moment_errs_15K = get_moment_vals(loglam, rest_fluxes15K, rest_ivars15K, Bindx15K, synth_B15K)
moments_25K, moment_errs_25K = get_moment_vals(loglam, rest_fluxes25K, rest_ivars25K, Bindx25K, synth_B25K)

print(moments_10K.shape, moment_errs_10K.shape)
print(moments_15K.shape, moment_errs_15K.shape)
print(moments_25K.shape, moment_errs_25K.shape)

In [ ]:
def plot_synthesized_best(synthB, rest_fluxes, Binx, log_lambda, moments, moment_errs, lines, temp, K = 12):
    foo = np.argsort(moments[:,0, 0] / moment_errs[:, 0, 0])[::-1] # descending 
    f = plt.figure(figsize=(12, 12))
    wave = 10 ** log_lambda

    offset = 0.5
    for i in range(K):
        f.gca().plot(wave, rest_fluxes[Binx[foo[i]]] + i * offset, color="k")
        f.gca().plot(wave, synthB[foo[i]] + i * offset, color="r")

    for i, line in enumerate(lines_balmer):
        if i == 0: 
            plt.axvline(line, lw=1, alpha=0.5, color="royalblue", label = "Balmer")
        else:
            plt.axvline(line, lw=1, alpha=0.5, color="royalblue")

    for i, line in enumerate(lines_HE_I):
        if i == 0: 
            plt.axvline(line, lw=1, alpha=0.5, color="hotpink", label = "HE I")
        else:
            plt.axvline(line, lw=1, alpha=0.5, color="hotpink")

    for i, line in enumerate(lines_HE_II):
        if i == 0: 
            plt.axvline(line, lw=1, alpha=0.5, color="green", label = "HE II")
        else:
            plt.axvline(line, lw=1, alpha=0.5, color="green")
            
    #plt.xlim(4500, 5000)
    plt.xlim(H_ALPHA - 200, H_ALPHA + 200)
    #plt.xlim(4000, 5000)
    plt.xlabel("Wavelength (angstrom)")
    plt.legend()
    halpha_lo = 10**(log_H_ALPHA - H_delta_loglam_line)
    halpha_hi = 10**(log_H_ALPHA + H_delta_loglam_line)
    plt.axvspan(halpha_lo, halpha_hi, color='royalblue', alpha=0.3)
    
    hbeta_lo = 10**(log_H_BETA - H_delta_loglam_line)
    hbeta_hi = 10**(log_H_BETA + H_delta_loglam_line)
    plt.axvspan(hbeta_lo, hbeta_hi, color='royalblue', alpha=0.3)
    
    if temp == 25:
        plt.title("Highest SNR synthesized spectra with Teff > 25000K")
    if temp == 15:
        plt.title("Highest SNR synthesized spectra with 15000 < Teff < 25000K")
    if temp == 10:
        plt.title("Highest SNR synthesized spectra with Teff > 10000K")
    plt.show()
   
    plt.close()

In [ ]:
plot_synthesized_best(synth_B25K, rest_fluxes25K, Bindx25K, loglam, moments_25K, moment_errs_25K, lines, temp = 25)

In [ ]:
plot_synthesized_best(synth_B15K, rest_fluxes15K, Bindx15K, loglam, moments_15K, moment_errs_15K, lines, temp = 15)

In [ ]:
plot_synthesized_best(synth_B10K, rest_fluxes10K, Bindx10K, loglam, moments_10K, moment_errs_10K, lines, temp = 10)

In [ ]:
#or every statistic?
def get_EW_for_every_spectra(spectra_df, synB10K, synB15K, synB25k):

    temps = spectra_df['Teff_fit'].to_numpy()
    print(temps.shape)
    nana_hash = spectra_df['NANA_HASH'].to_numpy()
    print(nana_hash.shape)
    
    for i, (nana, temp) in enumerate(zip(nana_hash, temps)):
        
        if temp > 25000:
            if nana%2 == 0: #(then it was og in A), so you synthesize using B model 

                print(synB25K.shape)
                print("using synth_B")
                print(temp)
            
            if nana%2 == 1: #(then it was og in B), so you syntehsize using A model
                print()
                # synthesize
                # once you synthesize get the relevant momentA and moment errsAkla,m.,
        elif  temp >= 15000:
            if nana%2 == 0: #(then it was og in A), so you synthesize using B model 
                print("using synth_B")
                print(synB15K.shape)
                print(temp)
            
            if nana%2 == 1: #(then it was og in B), so you syntehsize using A model
                print()
                # synthesize
                # once you synthesize get the relevant momentA and moment errsAkla,m.,
        elif 15000 > temp >= 1000:
            
            if nana%2 == 0: #(then it was og in A), so you synthesize using B model
                print()

            
            if nana%2 == 1: #(then it was og in B), so you syntehsize using A model
                # synthesize
                # once you synthesize get the relevant momentA and moment errsAkla,m.
                print()
        if i == 45:
            assert False

In [ ]:
#hogg doesn't liek that these numbers are written up

#make dictionary for hashes
#create a train dictionary it will tell you which model to train 
#create a test dictionary 
get_EW_for_every_spectra(spectra, synth_B10K, synth_B15K, synth_B25K)


#top of code will have mask list
#table or dictionary (name description)
#synthesize all six models
#and rpdocue badness of fit and make the scatter plot


In [ ]:
#write the training step once,
#for loop is setting up variables for runnning one line


#do six synthesize instead instead of 100,000. Group together the relevant temperature 
#build six indexes over the six cases
#loop for temperature cuts

#for temp in temp cut